this notebook contains the multivariate analysis of the crime types for the eucludian approach.

# 1.0 Load packages

In [ ]:
# load the packages
import pandas as pd
import json
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests
from statsmodels.discrete.discrete_model import NegativeBinomial
import warnings

In [ ]:
# suppress RuntimeWarnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
import gc
gc.collect()

In [ ]:
# import boundary data
bradford_boundary = gpd.read_file('bradford_boundary.geojson')

In [ ]:
# load the data
df = gpd.read_file('crime_nearest_school.geojson', use_arrow=True)

# 2.0 Buffer separation

In [ ]:
# load main cleaned data
crimes_df1 = df[df['distance_to_school_m'] <= 250].copy()
crimes_df2 = df[df['distance_to_school_m'] > 250].copy()
crimes_df3 = df[df['distance_to_school_m'] <= 500].copy()
crimes_df4 = df[df['distance_to_school_m'] > 500].copy()
crimes_df5 = df[df['distance_to_school_m'] <= 750].copy()
crimes_df6 = df[df['distance_to_school_m'] > 750].copy()
crimes_df7 = df[df['distance_to_school_m'] <= 1000].copy()
crimes_df8 = df[df['distance_to_school_m'] > 1000].copy()

In [ ]:
# track the buffer distance
crimes_df1['buffer_m'] = '250'
crimes_df2['buffer_m'] = '250+'
crimes_df3['buffer_m'] = '500'
crimes_df4['buffer_m'] = '500+'
crimes_df5['buffer_m'] = '750'
crimes_df6['buffer_m'] = '750+'
crimes_df7['buffer_m'] = '1000'
crimes_df8['buffer_m'] = '1000+'

In [ ]:
# concatenante the incidentypes
crimes = pd.concat(
    [crimes_df1, crimes_df2, crimes_df3, crimes_df4,
     crimes_df5, crimes_df6, crimes_df7, crimes_df8],
    ignore_index=True)

Recall that the only significant crime types were 'violence without injury', 'public order offences', 'violence with injury'. I will filter only these crimes for the multivariate analyses.

In [ ]:
sig_crimes = crimes[crimes['offence'].isin([
    'Violence without injury', 'Public order offences',
    'Violence with injury'])]

In [ ]:
sig_crimes.head()

# 3.0 Model selection

In [ ]:
sig_crimes.season.unique()

In [ ]:
# pivot teh table
wide_counts = pd.pivot_table(
    data=sig_crimes,
    index=['year&week', 'term_time_type', 'buffer_m'],
    columns='offence',
    values='season',
    aggfunc='count',
    fill_value=0
)

In [ ]:
# compute mean and variance
# loop through buffer
all_buffer_results = []
unique_buffers = sig_crimes['buffer_m'].unique()

for current_buffer in unique_buffers:
    # Isolate the rows belonging to the current buffer zone
    buffer_subset = wide_counts.xs(current_buffer, level='buffer_m')
    
    # Calculate localized mean and variance across the timeline rows
    buffer_means = buffer_subset.mean()
    buffer_variances = buffer_subset.var()
    
    # Compile the results for this specific distance band
    buffer_df = pd.DataFrame({
        'crime_type': buffer_means.index,
        'mean_count': buffer_means.values,
        'variance': buffer_variances.values,
        'buffer_distance': current_buffer
    })
    
    all_buffer_results.append(buffer_df)

In [ ]:
# put into a table
dispersion = pd.concat(all_buffer_results, axis=0)

In [ ]:
# calculate teh ratio as a way to identify dispersion
dispersion['dispersion_ratio'] = dispersion['variance'] / dispersion['mean_count']
dispersion = dispersion.sort_values(by='dispersion_ratio', ascending=False)

In [ ]:
pd.set_option('display.max_rows', None)
dispersion

I will only use the Negative Binomial model

## 3.1 Build the table

In [ ]:
# get the dates
calendar = (
    sig_crimes[['date_only', 'term_time_type']]
    .drop_duplicates())

In [ ]:
# get the crimes and the buffer
crime_table = pd.DataFrame({
    'offence': sig_crimes['offence'].unique()})

buffer_table = pd.DataFrame({
    'distance_comparison': sig_crimes['buffer_m'].unique()})

In [ ]:
# merge the crimes and distances to the daily
sig_df = (
    calendar.merge(crime_table, how='cross')
    .merge(buffer_table, how='cross'))

In [ ]:
# count crimes at each day and buffer
observed_counts = (
    sig_crimes
    .groupby(['date_only', 'offence', 'term_time_type', 'buffer_m'])
    .size()
    .reset_index(name='crime_count'))

In [ ]:
# rename buffer to fit teh sig_df table
observed_counts = observed_counts.rename(
    columns={'buffer_m': 'distance_comparison'}
)

In [ ]:
# add the counts
sig_df = sig_df.merge(
    observed_counts,
    on=['date_only', 'offence', 'term_time_type', 'distance_comparison'],
    how='left')

In [ ]:
# fill no crime with 0
sig_df['crime_count'] = (
    sig_df['crime_count']
    .fillna(0)
    .astype(int)
)

In [ ]:
# encode day and distance
sig_df['is_school_day'] = sig_df['term_time_type'].map({
    'Term time': 1,
    'holiday': 0
})

sig_df['is_distance_within'] = (
    sig_df['distance_comparison']
    .str.endswith('+')
    .map({True: 1,
          False: 0})
)

# 4.0 Negative Binomial Regression

Daily crime counts are modelled using Negative Binomial regression. This specification is appropriate for count outcomes and accommodates overdispersion in the daily crime-count distribution. The model includes proximity to a secondary school, school-day status, and their interaction. Cluster-robust standard errors are estimated by date to account for dependence between proximity groups observed on the same day.

he model examines whether crime counts differ according to:

proximity to a secondary school;
whether the date is a school day or holiday; and
the interaction between proximity and school-day status.

The count model is:

$$ \log(\mu_i) = \beta_0 + \beta_1\text{Within}_i + \beta_2\text{SchoolDay}_i + \beta_3(\text{Within}_i\times\text{SchoolDay}_i) $$

where:

$$ \mu_i = E(Y_i|X_i) $$

and:

\(Y_i\) = observed daily crime count for observation \(i\)
\(\mu_i\) = expected daily crime count
\(\beta_0\) = intercept: expected log crime count for the reference group
\(\beta_1\) = effect of being within the specified proximity threshold rather than further away
\(\beta_2\) = effect of a school day rather than a holiday
\(\beta_3\) = interaction effect, indicating whether the within-versus-further relationship changes on school days
Within = 1 for observations within the specified buffer and 0 for further away
SchoolDay = 1 for school days and 0 for holidays

The reference category is therefore:

Further away + Holiday

In [ ]:
# model formula
formula = 'crime_count ~ C(is_distance_within) * is_school_day'

In [ ]:
# get the offences and buffer distances
offences= sorted(
    sig_df['offence']
    .dropna()
    .unique())

# Each number identifies one distance pair
buffer_distances = [250, 500, 750, 1000]

In [ ]:
# Store models by offence, then distance pair
offence_models = {}

for offence in offences:
    offence_models[offence] = {}

    for distance in buffer_distances:
        # Select both sides of this pair
        pair = [str(distance), f"{distance}+"]

        model_data = sig_df[
            (sig_df["offence"] == offence) &
            (sig_df["distance_comparison"].astype(str).isin(pair))
        ].copy()

        # Fit and store the model for this complete pair
        model = NegativeBinomial.from_formula(formula, data=model_data)
        result = model.fit(method="bfgs", maxiter=1000, disp=False, cov_type="cluster",
                           cov_kwds={"groups": model_data["date_only"]})
        offence_models[offence][distance] = result

        print(f"\nOffence: {offence} | Distance pair: {distance} and {distance}+")
        print(result.summary())

Only Violence with injury 250 vs 250+ (p < 0.05)

Extract the reults from the coefficients

In [ ]:
# define interest
terms = {
    "C(is_distance_within)[T.1]": "within distance coeff",
    "is_school_day": "school day coeff",
    "C(is_distance_within)[T.1]:is_school_day": "interaction coeff"
}

In [ ]:
# initialise list
result_rows = []

for offence, distance_models in offence_models.items():
    for distance, result in distance_models.items():
        for model_term, label in terms.items():
        
            # Extract this term's coefficient and confidence interval
            coefficient = result.params[model_term]
            lower, upper = result.conf_int().loc[model_term]

            result_rows.append({
                "offence": offence,
                "distance": distance,
                "term": label,
                "coefficient": coefficient,
                "irr": np.exp(coefficient),
                "irr_ci_lower": np.exp(lower),
                "irr_ci_upper": np.exp(upper),
                "p_value": result.pvalues[model_term],
                "n_observations": result.nobs
            })

In [ ]:
# put into a datframe and sort
model_results_df = (
    pd.DataFrame(result_rows)
    .sort_values(["offence", "distance"])
    .reset_index(drop=True)
)

# model_results_df.round(4)

In [ ]:
# save model results
model_results_df.to_csv('euclidian_crimetype_model_results.csv')

statistical inference: adjust the p value for interaction

In [ ]:
# Start with empty adjustment columns
model_results_df["adjusted_p_value"] = np.nan
model_results_df["significant_after_bh"] = False

In [ ]:
# Select interaction rows with valid p-values
interaction_mask = (
    (model_results_df["term"] == "interaction coeff") &
    model_results_df["p_value"].notna()
)

In [ ]:
# Adjust the interaction p-values across all offence × distance models
if interaction_mask.any():
    significant, adjusted_p, _, _ = multipletests(
        model_results_df.loc[interaction_mask, "p_value"],
        alpha=0.05, method="fdr_bh")

    # Put the results back into their original rows
    model_results_df.loc[interaction_mask, "adjusted_p_value"] = adjusted_p
    model_results_df.loc[interaction_mask, "significant_after_bh"] = significant

In [ ]:
# view only interaction
display(
    model_results_df.loc[
        model_results_df["term"] == "interaction coeff",
        ["offence", "distance", "p_value",
         "adjusted_p_value", "significant_after_bh"]
    ]
)

After adjustment of p-value, there was no significance.

## 4.1 Model evaluation

In [ ]:
evaluation_rows = []

for offence, distance_models in offence_models.items():
    for distance, result in distance_models.items():
        # Observed and fitted daily crime counts for rows used by this model
        observed = np.asarray(result.model.endog)
        predicted = np.asarray(result.predict())

        # NB2 variance: mean + alpha × mean²
        alpha = result.params["alpha"]
        variance = predicted + alpha * predicted**2

        # Pearson dispersion uses the observation-level residuals
        pearson_residuals = (observed - predicted) / np.sqrt(variance)
        pearson_dispersion = (
            np.sum(pearson_residuals**2) / result.df_resid
        )

        # Probability of zero counts under the fitted NB2 model
        predicted_zero_percent = (
            (1 + alpha * predicted) ** (-1 / alpha)
        ).mean() * 100

        evaluation_rows.append({
            "offence": offence,
            "distance": distance,
            "n_observations": result.nobs,
            "converged": result.mle_retvals["converged"],
            "alpha": alpha,
            "pearson_dispersion": pearson_dispersion,
            "mae": np.mean(np.abs(observed - predicted)),
            "rmse": np.sqrt(np.mean((observed - predicted)**2)),
            "observed_zero_percent": np.mean(observed == 0) * 100,
            "predicted_zero_percent": predicted_zero_percent
        })

In [ ]:
# put into a dataframe
evaluation_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values(["offence", "distance"])
    .reset_index(drop=True)
)

display(evaluation_df)

All models converged successfully and displayed Pearson dispersion values close to one, suggesting that the negative binomial specification adequately handled count variability. Violence with injury demonstrated the strongest predictive performance, with the lowest MAE and RMSE values and close agreement between observed and predicted zero proportions. Public order offence models also performed well. Violence without injury exhibited higher prediction errors (between 1.03 and 1.22) and mild residual overdispersion, indicating greater unexplained heterogeneity, although overall model fit remained acceptable.

In [ ]:
# save result
evaluation_df.to_csv('euclidan_crimetype_evaluation_results.csv')